# Final Analysis of 39 Penalty Attempts
This notebook reads the consolidated candidate bundle and validates its IDs and hashes. It does not run model inference or alter saved results. All comparisons concern inspected development records; candidate availability and reference agreement are not held-out accuracy.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'src/eda_analysis.py').is_file(), 'Run from the project root or notebooks directory'
sys.path.insert(0, str(root))
from src.eda_analysis import load_analysis, summarize_analysis, make_figures

features, evaluation, provenance = load_analysis(root / 'outputs/final_analysis/features_39.csv')
summary = summarize_analysis(features, evaluation)
display(pd.Series({key: evaluation.get(key) for key in ['total_events', 'held_out_test', 'accuracy_validated', 'reference_sources_independently_verified']}))
print('Artifact integrity verified. Raw-video verification:', provenance.get('raw_source_verification'))
display(features.head())

## Q1: Which Features Have Candidate Evidence Across Matches?
A candidate, a recorded reference, and a verified measurement have different meanings. Missing values and unknown categories remain in the event denominator. Camera intervals do not establish camera-class accuracy; silhouette proxies do not establish physical height.

In [ ]:
display(pd.Series(summary['candidate_coverage'], name='Available events / 39'))
display(pd.DataFrame({match: item['available'] for match, item in summary['coverage_by_match'].items()}))
display(pd.Series(summary['validated_coverage'], name='Verified or validated events'))

## Q2: What Do Step Candidates, Parameter Sensitivity, and Censoring Show?
Counts describe visible ankle-signal candidates, not confirmed foot landings. An absent partial-runup flag does not prove complete coverage. Fixed-parameter ranges are sensitivity diagnostics rather than confidence intervals; the data do not support estimating a partial-runup effect or step-count accuracy.

In [ ]:
display(pd.Series(summary['steps']))
display(features.loc[features['partial_runup'].fillna(False), ['clip_id', 'step_count_candidate', 'step_count_verified', 'partial_runup']])

## Q3: How Do Decided Ball Rules Compare With Recorded External References?
The confusion table uses only cases with a decided rule and a recorded reference. Abstentions remain visible. The reference-source evidence has not been independently reverified in this consolidation, and the cohort is not held out. Covered reference agreement must be reported alongside coverage.

In [ ]:
comparison = summary['ball_reference_comparison']
display(pd.Series({key: value for key, value in comparison.items() if key != 'confusion_matrix'}))
display(pd.DataFrame.from_dict(comparison['confusion_matrix'], orient='index').rename_axis('External reference').rename_axis('Mapped rule candidate', axis=1))
display(features.loc[features['clip_id'].isin(comparison['conflict_ids']), ['clip_id', 'ball_rule_outcome', 'reference_outcome', 'review_required']])

In [ ]:
import matplotlib.pyplot as plt
figures = make_figures(features, summary)
for name, figure in figures.items():
    display(figure)
    plt.close(figure)

## Limits and Next Evidence
The analysis does not create independent annotations, calibrated ground-plane paths, physical goalkeeper heights, or verified final ball positions. It does not establish that the rules generalize to new footage. The separate ten-clip kicking-foot comparison is development evidence after user feedback. Full-event human review and a whole-match evaluation reserved before tuning are needed for future accuracy claims. Methods and provenance boundaries are documented in `docs/FINAL_ANALYSIS_METHODS.md`.